In [1]:
from autogluon.tabular import TabularDataset, TabularPredictor
import pandas as pd

/home/alexandre/Projects/ML/MLvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = TabularDataset("/home/alexandre/Projects/ML/KS/Titanic/data/train.csv")
test  = TabularDataset("/home/alexandre/Projects/ML/KS/Titanic/data/test.csv")

LABEL   = "Survived"
ID_COL  = "PassengerId"

In [ ]:
predictor = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path="AutogluonModels_titanic"
)

# Hyperparamètres : mélange ML + deep learning + texte
hyperparameters = {

    # Arbres
    "XGB": {  # XGBoost GPU
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
    },
    "GBM": {  # LightGBM GPU (nécessite LightGBM compilé GPU)
        "device": "gpu",
    },
    "CAT": {  # CatBoost GPU
        "task_type": "GPU",
        "devices": "0",
    },

    # Réseaux
    "NN_TORCH": {},   # utilise le GPU automatiquement via AG_ARGS_FIT
    "AG_TEXT_NN": {}, # idem (exploite Name/Ticket/Cabin/… en texte)

    # Les suivants restent CPU (pas critique)
    "RF": {}, "XT": {}, "LR": {}, "KNN": {}
}

In [ ]:
predictor.fit(
    train_data=train,
    presets="best_quality",
    time_limit=None,          # mets une limite en secondes si besoin
    hyperparameters=hyperparameters,
    auto_stack=True,
    num_bag_folds=5,          # bagging 
    num_stack_levels=2,       # empilement niveau 1 (peut tester 2)
    refit_full=True,           # réentraine sur 100% des données au final
    ag_args_fit={"num_gpus": 1}  # alloue le GPU aux modèles qui savent l'utiliser
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          32
Memory Avail:       13.66 GB / 15.51 GB (88.1%)
Disk Space Avail:   864.87 GB / 1006.85 GB (85.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect sta

AssertionError: Unknown model type specified in hyperparameters: 'AG_ARGS_FIT'. Valid model types: ['RF', 'XT', 'KNN', 'GBM', 'CAT', 'XGB', 'REALMLP', 'NN_TORCH', 'LR', 'FASTAI', 'AG_TEXT_NN', 'AG_IMAGE_NN', 'AG_AUTOMM', 'FT_TRANSFORMER', 'TABICL', 'TABM', 'TABPFNMIX', 'TABPFNV2', 'MITRA', 'FASTTEXT', 'ENS_WEIGHTED', 'SIMPLE_ENS_WEIGHTED', 'IM_RULEFIT', 'IM_GREEDYTREE', 'IM_FIGS', 'IM_HSTREE', 'IM_BOOSTEDRULES', 'DUMMY']

In [ ]:
preds = predictor.predict(test)          # 0/1
submission = pd.DataFrame({ID_COL: test[ID_COL], LABEL: preds.astype(int)})
submission.to_csv("submission_autogluon_2.csv", index=False)
print("submission.csv écrit.")
print(submission.head(10))

submission.csv écrit.
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
5          897         0
6          898         0
7          899         0
8          900         1
9          901         0
